### API Pull (required):

In [ ]:
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

  Using cached yfinance-1.6.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached curl_cffi-0.16.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (17 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached peewee-4.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached protobuf-7.35.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
Using cached yfinance-1.6.0-py3-none-any.whl (148 kB)
Using cached curl_cffi-0.16.0-cp310-abi3-macosx_11_0_arm64.whl (2.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 68.7 MB/s  0:00:00
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
Using cached peewee-4.3.0-py3-none-any.whl (179 kB)
Using cached protobuf-7.35.1-cp310-abi3-macosx_10_9_universal2.whl (433 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [yfinance]7/8 [yfinance]


In [3]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

In [4]:
def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

In [5]:
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")
safe_stamp

<function __main__.safe_stamp()>

In [8]:
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/jenyunting/Desktop/bootcamp_yunting_jen/homework/homework4/notebooks

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [12]:
SYMBOL = "NVDA"
from dotenv import load_dotenv


load_dotenv()
ALPHA_KEY = os.getenv("API_KEY")
DATA_RAW = pathlib.Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

use_alpha = bool(ALPHA_KEY)
print(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    if not key:
        # Alpha Vantage replies HTTP 200 with a prose blob - not an error status - when
        # the free tier's daily cap is hit or the endpoint has moved to premium, so
        # raise_for_status() above sees nothing wrong. Say so and fall back.
        print("Alpha Vantage returned no series:", str(list(js.values())[0])[:150])
        use_alpha = False

if use_alpha:
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

if not use_alpha:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d", auto_adjust=False,
                          multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','close'], dtypes_map={'date':'datetime64[ns]','close':'float'})
print(msgs)

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

MFOUPD11R87HXEAZ
Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Saved: ../data/raw/api_source-alpha_symbol-NVDA_20260817-092118.csv


### Scrape a Small Table (required):

In [13]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: instructor@example.edu)"}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', class_='wikitable')   # the components table
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    # assume first row is header
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print("Scrape failed (demoing with inline HTML).", e)
    html = """
    <table>
      <tr><th>Ticker</th><th>Price</th></tr>
      <tr><td>AAA</td><td>101.2</td></tr>
      <tr><td>BBB</td><td>98.7</td></tr>
    </table>
    """
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for tr in soup.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td','th'])]
        if cells:
            rows.append(cells)
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')

msgs2 = validate_df(df_scrape, required_cols=list(df_scrape.columns), dtypes_map={})
print(msgs2)

fname2 = safe_filename(prefix="scrape", meta={"site": "wikipedia", "table": "djia"})
out_path2 = DATA_RAW / fname2
df_scrape.to_csv(out_path2, index=False)
print("Saved:", out_path2)

{'na_total': 'Total NA values: 0'}
Saved: ../data/raw/scrape_site-wikipedia_table-djia_20260817-100008.csv


### Documentation

- **API Source:** Alpha Vantage API  
  - Endpoint: `https://www.alphavantage.co/query`
  - Ticker: `NVDA`
  - Function: `TIME_SERIES_DAILY`
  - Output size: `compact`
  - Data format: JSON
  - If Alpha Vantage is unavailable or rate-limited, the notebook falls back to Yahoo Finance (`yfinance`).

- **Scrape Source:** Wikipedia – List of S&P 500 Companies  
  - URL: `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`
  - Table: S&P 500 component companies table (`wikitable`).

- **Assumptions & Risks:**
  - Alpha Vantage may be subject to API rate limits or usage restrictions.
  - The API response schema or endpoint may change in the future.
  - Web scraping depends on the current HTML structure of the Wikipedia page; changes to the table structure or CSS class may cause the scraper to fail.
  - Missing or unexpected data may require additional validation or cleaning.

- **Environment Security:** The API key is stored in `.env`, and `.env` is not committed to GitHub.